# UT3: Programación funcional con R

La programación funcional, en un sentido amplio, es aquella en que determinadas funciones1 admiten otras como argumento. En R, las funciones son ciudadanos de primera clase, lo que significa que pueden ser tratadas como cualquier otro objeto: pueden ser asignadas a variables, pasadas como argumentos a otras funciones y devueltas como valores desde funciones. Esto permite un estilo de programación más abstracto y modular, facilitando la reutilización del código y la creación de funciones más generales.     

Las funciones denominadas **de alto orden** en R son aquellas que toman otras funciones como argumentos o devuelven funciones como resultados.

Por ejemplo, la función `sapply()` es una función de alto orden que aplica una función a los márgenes de un array o matriz. Aquí hay un ejemplo simple:

In [1]:
cuadrado.propio <- function(x) {
  if (x < 5) x ^ 2 else -x ^ 2
}
sapply(1:10, cuadrado.propio) # Aplicar la función a cada elemento del vector 1:10

 [1]    1    4    9   16  -25  -36  -49  -64  -81 -100

La programación funcional es sumamente poderosa y sugiere permite programar de la siguiente manera:

- Crear funciones pequeñas y simples que resuelven un problema pequeño y acotado
- Aplicar esas funciones a grupos homogéneos de valores.   

En el ejemplo de más arriba hemos construido una función, cuadrado.raro, y con la función `sapply` se ha aplicado a una lista de valores homogéneos, los números del 1 al 10.

Hay muchas funciones en R que admiten otras como argumento. Algunas de las más corrientes son:

- `sapply` y `lapply` (que son casi la misma)
- `tapply`
- `apply` y `mapply`
- Las funciones `ddply`, `ldply`, etc. del paquete `plyr`

Dos ejemplos de usos muy habituales de estas funciones son:
```R
lapply(iris, class)
sapply(iris, length)
```   
En el primer caso, se obtiene la clase de cada una de las columnas del conjunto de datos `iris`. En el segundo, se obtiene la longitud (número de elementos) de cada columna. Estas funciones aprovechan la naturaleza de las listas en R, ya que los data frames son esencialmente *listas de columnas*.

### Las funciones `lapply`, `sapply` y `split`  

La función `lapply` está estrechamente relacionada con las listas. 
La función `lapply` es una función de orden superior que aplica una función a cada elemento de una lista o vector. Por ejemplo:   

```R
lapply(airquality[, 1:4], mean, na.rm = TRUE) # na.rm = TRUE es un argumento adicional que hace que se ignoren los valores NA. rm significa "remove" (eliminar)
```

Esta función calcula la media de cada una de las cuatro primeras columnas de *airquality*. El argumento adicional, `na.rm = TRUE`, se pasa a la función `mean`.

La función `lapply` opera sobre una lista o vector y devuelve, siempre, una lista. Está relacionada con la función `sapply`, que es una versión simplificada de `lapply`. Es decir, `sapply` aplica la función `lapply` y estudia la salida. Cuando entiende que dicha salida admite una representación menos aparatosa que una lista, la simplifica. Por ejemplo:
```bash

sapply(airquality[, 1:4], mean, na.rm = TRUE)

     Ozone    Solar.R       Wind       Temp 
 42.129310 185.931507   9.957516  77.882353 
```

En el caso anterior, en lugar de una lista, como hace `lapply`, `sapply` devuelve una representación más natural: un vector de números.

Tanto `lapply` como `sapply` son ***maps***, es decir, *funciones que aplican una función a cada elemento de una lista o vector*. En muchos casos habituales, no hace falta usar estas funciones porque muchas están vectorizadas. Por ejemplo:

```bash
sqrt(1:5)

[1] 1.000000 1.414214 1.732051 2.000000 2.236068

```
Pero si `sqrt` no estuviese vectorizada, aún podríamos aplicársela a un vector haciendo, por ejemplo:
```bash
sapply(1:5, sqrt)

[1] 1.000000 1.414214 1.732051 2.000000 2.236068
```
Obviamente, `lapply` y `sapply` permiten aplicar cualquier función, incluidas las definidas por los usuarios.

Finalmente, también es muy útil la función `split`, que permite partir una tabla en trozos de acuerdo con un vector que define los grupos. Por ejemplo: 
```R

tmp <- split(iris, iris$Species)
```

- Ejercicio: Investiga el objeto `tmp`: ¿qué longitud tiene? ¿qué contiene cada uno de sus componentes?
- Ejercicio: Usa las funciones `lapply` y `sapply` para mostrar la dimensión de cada una de las tablas que contiene `tmp`.

La función `split` puede servir para operar en subtablas en combinación con `lapply`. Por ejemplo, para calcular la media de cada columna de `iris` por especie, podemos hacer:
```R
sapply(split(iris[, 1:4], iris$Species), colMeans)
```


- Ejercicio: Usa split para partir iris en dos subtablas al azar con el mismo número de filas.
- Ejercicio: Usa split para partir iris cinco subtablas, cada una de ellas con 30 filas distintas.

Es conveniente recordar aquí que si consultas la ayuda de las funciones listadas más arriba verás que suelen incluir un argumento especial, `...` que permite pasar argumentos adicionales a la función a la que llaman. Por ejemplo, en `sapply(X, FUN, ...)`, los argumentos adicionales en `...` se pasan a la función `FUN`. Esto es útil cuando `FUN` requiere parámetros adicionales para su ejecución.

In [2]:
# Ejemplo de uso de sapply con una función con parámetros adicionales
potencia <- function(base, exponente) {
  base ^ exponente
}
bases <- 1:5
exponente <- 3
sapply(bases, potencia, exponente) # Aplicar la función potencia a cada base con exponente fijo 

[1]   1   8  27  64 125

## Funciones anónimas   

Las funciones que se han usado hasta ahora son de dos tipos: o existen en R o se han definido previamente. Pero en ocasiones es conveniente usar ***funciones anónimas*** de esta manera:  

```R
sapply(1:10, function(x) if(x < 5) x^2 else -x^2)
```

Conviene particularmente cuando la función solo se usa una única vez. Las ***funciones anónimas***, debidamente usadas, confieren brevedad y expresividad al código.

En definitiva, una **función anónima** es una *función sin nombre* que se define en el lugar donde se necesita, generalmente como argumento para otra función. Se crean utilizando la palabra clave `function` seguida de los parámetros entre paréntesis y el cuerpo de la función entre llaves.

## Map, Reduce y otros conceptos de programación funcional   

Existen dos operaciones fundamentales en programación funcional. La primera es ***map*** y consiste en aplicar una función a todos los elementos de una lista o vector.

#### Función *Map*   
Aplica una función a cada elemento de un vector o lista, y devuelve un nuevo vector o lista con los resultados. Es como si se dispusiera de una máquina que toma cada elemento de una colección de datos, lo procesa con la función que se le indique, y coloca el resultado en una nueva colección.

In [18]:
library(purrr)
# Crear un vector de números
numeros <- c(1, 2, 3, 4, 5)

# Calcular el cuadrado de cada número usando una función anónima
cuadrados <- map(numeros, function(x) x^2)
cuadrados

[[1]]
[1] 1

[[2]]
[1] 4

[[3]]
[1] 9

[[4]]
[1] 16

[[5]]
[1] 25


In [19]:
# También podemos usar una función predefinida
raices_cuadradas <- map(numeros, sqrt)

# Mostrar el resultado
raices_cuadradas

[[1]]
[1] 1

[[2]]
[1] 1.414214

[[3]]
[1] 1.732051

[[4]]
[1] 2

[[5]]
[1] 2.236068



También existen otras operaciones que pueden considerarse (y se consideran) operaciones **map**, como la operación que hemos realizado más arriba:

```R
sapply(1:10, function(x) if(x < 5) x^2 else -x^2)
```
Ese código aplica al vector `1:10` la función anónima que se le pasa a `sapply` como segundo argumento. Aunque en muchos lenguajes de programación existe una función `map` explícita (con ese nombre), en R hay varias: además de `sapply`, también están `lapply` o `apply`, como ya se ha visto previamente.   

La vectorización vista previamente es un mecanismo implícito para realizar `maps; p.e.:
```R
sqrt(1:10)
```
aplica la función `sqrt` a cada elemento de su argumento.

La otra gran operación de la programación funcional es `Reduce`. Consiste en aplicar una operacion binaria (p.e., la que suma dos números) a una lista de valores iterativamente.       
Dicho de otra forma, `Reduce`combina los elementos de un vector o lista aplicando una función de forma acumulativa. Es como si se tuviera una máquina que toma dos elementos de una colección, los combina usando la función que se le indique, y luego combina el resultado con el siguiente elemento, y así sucesivamente hasta que se combinan todos los elementos.


In [22]:
# Crear un vector de números
numeros <- c(1, 2, 3, 4, 5)

# Calcular la suma de los números usando reduce()
suma <- reduce(numeros, `+`)

# Mostrar el resultado
print(suma)

[1] 15


In [23]:

# Calcular el producto de los números
producto <- reduce(numeros, `*`)

# Mostrar el resultado
print(producto)


[1] 120


En R se pueden realizar *reduces* explícitamente:
```R
Reduce(function(a, b) a + b, 1:10)
```
La función anónima anterior admite dos argumentos, `a` y `b` y los suma. Dentro de `Reduce`, la función anónima suma los dos primeros elementos, al resultado le suma el tercero, al resultado el cuarto, etc., hasta proporcionarnos la suma total. De nuevo, es frecuente poder realizar estas operaciones implícitamente. Por ejemplo, usando `sum`:
```R
sum(1:10)
```

- Ejercicio: Crea el objeto `x <- split(iris, iris$Species)`, que es una lista de tres tablas. Usa `lapply` o `sapply` para examinarlas: dimensión, nombres de columnas, etc.   

Operaciones tales como
```R
sum(sqrt(1:10))
```
son, por lo tanto, los famosos `mapreduces` popularizados por las herramientas de big data.

Una operación relacionada con `map` y muy frecuente en R es `replicate`. Esta función permite llamar repetidamente a una función (o bloque de código) para, de forma muy habitual, realizar simulaciones:

In [ ]:
simula <- function(n, lambda = 4, mean = 5){
  n.visitas <- sum(rpois(n, lambda)) # rpois genera números aleatorios de una distribución de Poisson, para generar el número de visitas
  sum(rnorm(n.visitas, mean = mean)) # rnorm genera números aleatorios de una distribución normal, para obtener la suma de las visitas   
}

res <- replicate(1000, simula(10, lambda = 7)) # Realizar 1000 simulaciones usando replicate y la función simula, que generará la suma de visitas para n=10 y lambda=7
res # Resultado de las 1000 simulaciones

   [1] 409.9304 349.1546 270.3019 313.3368 356.7647 263.0556 405.2125 259.2464
   [9] 278.9274 315.1611 306.7899 405.6827 391.0620 382.2730 323.5550 272.7292
  [17] 432.7254 302.6229 325.8929 352.0475 371.3996 459.7678 334.5945 288.0989
  [25] 336.1357 428.2171 366.1638 309.1769 323.3107 382.8808 386.9358 349.2187
  [33] 284.6211 392.0283 323.0886 375.3516 363.9003 476.0078 349.9123 311.6465
  [41] 296.4458 291.1542 312.7270 373.7812 365.1271 324.2819 374.7452 423.3546
  [49] 401.2190 339.9810 408.4385 322.0348 361.3006 336.0531 284.2806 298.7342
  [57] 353.4011 343.9068 336.0609 375.4020 334.1276 293.9549 365.3372 400.1065
  [65] 422.7897 264.3228 297.5038 306.4219 399.0696 440.5869 347.3473 380.4787
  [73] 310.9645 340.5515 325.1695 317.4893 352.3735 334.5259 251.8807 382.8159
  [81] 312.4345 284.4460 349.0208 336.1406 412.2451 321.3766 317.7892 309.4148
  [89] 354.2887 379.1467 435.4685 421.5244 401.3053 404.6854 343.5394 398.1090
  [97] 373.7251 377.7415 251.4510 310.8107 328.5087 

La diferencia fundamental con `map` es que no opera sobre un vector: crea uno de la nada.

#### Función *keep()*    

Esta función realiza el filtrado de los elementos de un vector o lista que cumplen una condición. 

In [24]:
# Crear un vector de números
numeros <- c(1, 2, 3, 4, 5)

# Filtrar los números pares
pares <- keep(numeros, ~ . %% 2 == 0)

# Mostrar el resultado
pares

[1] 2 4

El símbolo `~` en las funciones de alto orden se utiliza para definir una *función anónima*. Esto significa que se está creando una función “sobre la marcha”, sin necesidad de darle un nombre explícito, como ya se ha visto.     
La parte que sigue al `~` es el cuerpo de esta función, donde se especifican las operaciones que se realizarán sobre cada elemento del vector o lista al que se aplica la función.    
El punto `.` se utiliza como un marcador de posición para referirse al elemento actual.

In [25]:
# Filtrar los números mayores que 3
mayores_que_3 <- keep(numeros, ~ . > 3)

# Mostrar el resultado
mayores_que_3  # Output: 4 5

[1] 4 5

#### Función *Filter()*   

Otra metaoperación básica en programación funcional es `filter`.    
Un filtro permite seleccionar aquellas observaciones (p.e., en una lista) que cumplen cierta condición.    
En R es posible implementar esta operación usando los corchetes. Por ejemplo, para seleccionar los múltiplos de tres en un vector:

In [5]:
x <- 1:20
x[x %% 3 == 0]

[1]  3  6  9 12 15 18

Usando `filter`:

In [6]:
Filter(function(i) i %%3 == 0, x)

[1]  3  6  9 12 15 18

Por último, y no menos importante, indicar que en R, debido a lo habitual de operar con tablas, son las operaciones basadas en otra operación funcional, el `groupby`. El `groupby` permite partir un objeto en trozos de acuerdo con un determinado criterio para operar a continuación sobre los subloques obtenidos.    
`tapply` es una función que implementa una versión básica del `groupby`. Mucho más potente y versátil que ella es la función `ddply` del paquete `plyr`. Esta función realiza tres operaciones:

- Parte una tabla convenientemente (`groupby`).
- Aplica (`map`) una función sobre cada subtabla.
- Recompone (`reduce`) una tabla resultante a partir de los trozos devueltos en el paso anterior.

La función `tapply` es muy similar a la función `apply`. En el siguiente bloque de código se muestra la sintaxis de la función y la descripción simplificada de cada argumento:

In [ ]:
tapply(X,               # Objeto divisible (matriz, data frame, ...)
       INDEX,           # Lista(s) de factores de la misma longitud que X
       FUN,             # Función que se aplicará a los factores (o NULL)
       ...,             # Argumentos adicionales para pasar a FUN
       default = NA,    # Si simplify = TRUE, es el valor de inicialización del array
       simplify = TRUE) # Si FALSE devuelve una lista

Se debe tener en cuenta que los tres primeros argumentos son los más habituales y que es común no especificar el nombre de los argumentos en las funciones de la familia `apply` debido a su simple sintaxis.

#### Ejemplo de uso:

In [ ]:
# Generación de datos de ejemplo
set.seed(2)

data_set <- data.frame(precio = round(rnorm(25, sd = 10, mean = 30)),
                       tipo = sample(1:4, size = 25, replace = TRUE),
                       tienda = sample(paste("Tienda", 1:4),
                                      size = 25, replace = TRUE))

head(data_set)

  precio tipo   tienda
1     21    2 Tienda 2
2     32    3 Tienda 3
3     46    4 Tienda 4
4     19    3 Tienda 4
5     29    1 Tienda 4
6     31    3 Tienda 4

In [8]:
# Almacenamos los valores como variables
precio <- data_set$precio
tienda <- data_set$tienda
tipo <- factor(data_set$tipo,
               labels = c("juguetes", "comida", "electrónica", "bebidas"))

Ahora, se usará la función `tapply` para calcular la media por tipo de objeto de las tiendas de la siguiente manera:

In [9]:
# Precio medio por tipo de producto
precios_medios <- tapply(precio, tipo, mean)
precios_medios

   juguetes      comida electrónica     bebidas 
   39.50000    30.33333    32.20000    29.33333 

Nótese que los argumentos de la función `tapply` deben tener la misma longitud. Se puede verificar con la función `length`. También debe tenerse en cuenta que la salida predeterminada es de clase “array”.

In [10]:
class(precios_medios)

[1] "array"

Así pues, si es necesario, es posible acceder a cada elemento de la salida especificando el índice deseado entre corchetes.

In [11]:
precios_medios[2] # 30.33333

  comida 
30.33333 

Sin embargo, la clase de salida puede ser de clase `list` (una lista) si se establece el argumento `simplify` como `FALSE`.

In [12]:
# Precio medio por tipo de producto
lista_precios_medios <- tapply(precio, tipo, mean, simplify = FALSE)
lista_precios_medios

$juguetes
[1] 39.5

$comida
[1] 30.33333

$electrónica
[1] 32.2

$bebidas
[1] 29.33333


En este caso, también es posible acceder a los elementos de salida con el signo $ y el nombre del elemento:


In [13]:
lista_precios_medios$juguetes # 39.5

[1] 39.5

##### Argumentos adicionales: Ignorar valores NA   

Si se supone que se tiene un data frame que contiene algunos valores `NA` en sus columnas.

In [14]:
# Añadiendo valores NA al conjunto de datos
data_set[1, 1] <- NA
data_set[2, 3] <- NA

# Precio medio por tienda
tapply(data_set$precio, data_set$tienda, mean)

Tienda 1 Tienda 2 Tienda 3 Tienda 4 
32.00000       NA 39.25000 33.14286 

Dentro de la función `tapply` se puede especificar argumentos adicionales de la función que se está aplicando después del argumento `FUN`. En este caso, la función `mean` permite especificar el argumento `na.rm` para eliminar los valores NA (por defecto este argumento es FALSE).

In [15]:
tapply(data_set$precio, data_set$tienda, mean, na.rm = TRUE)

Tienda 1 Tienda 2 Tienda 3 Tienda 4 
32.00000 33.50000 39.25000 33.14286 

Que es equivalente a:

In [16]:
f <- function(x) mean(x, na.rm = TRUE)
tapply(data_set$precio, data_set$tienda, f)

Tienda 1 Tienda 2 Tienda 3 Tienda 4 
32.00000 33.50000 39.25000 33.14286 

Por último, también se puede aplicar la función tapply a varias columnas (o variables de tipo factor) pasándolas a través de una lista.    

En el siguiente ejemplo, se va a aplicar la función tapply a los factores tipo y tienda para calcular el precio medio de los objetos por tipo y tienda:

In [17]:
# Precio medio por tipo de producto y tienda
tapply(precio, list(tipo, tienda), mean)

            Tienda 1 Tienda 2 Tienda 3 Tienda 4
juguetes          46 31.00000       49 36.66667
comida            26 30.33333       39       NA
electrónica       50 29.00000       32 25.00000
bebidas           22 40.00000       20 36.00000

Es posible observar que no se vendió comida en la Tienda 4, por lo que se devuelve un `NA`. Para sobrescribir este comportamiento, se puede asignar en el argumento `default` un valor para que aparezca en lugar de un NA. En este ejemplo, se decide establecerlo en 0.